# Menjalankan aplikasi klasifikasi genre musik di Google Colab

Notebook ini mengambil source code dari GitHub, menyimpan dataset dan model di Google Drive, lalu menjalankan aplikasi Streamlit di Colab.

**Sebelum mulai:** push source code project ke `main` di GitHub. Dataset FMA Small tidak diunggah ke GitHub; siapkan folder `fma_small` di `MyDrive` dengan audio pada pola `*/*.mp3`, serta `tracks.csv` atau `fma_metadata.zip` di folder utama. Contoh: `MyDrive/fma_small/000/000002.mp3` dan `MyDrive/fma_small/fma_metadata.zip`.

Pilih **Runtime → Run all**, lalu izinkan akses Google Drive saat diminta. Pelatihan model adalah langkah terpisah dan bisa memerlukan waktu lama.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Sofwanfadilah/appskripsi.git"
REPO_DIR = Path("/content/appskripsi")

if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

if not (REPO_DIR / "app.py").is_file():
    raise FileNotFoundError(
        "Source app.py belum ditemukan. Pastikan commit project sudah di-push "
        "ke GitHub, lalu mulai ulang runtime Colab dan jalankan sel ini lagi."
    )

os.chdir(REPO_DIR)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", "requirements.txt"],
    check=True,
)

## Hubungkan dataset dan model di Google Drive

Unggah dataset FMA Small ke `MyDrive/fma_small`. Sel ini membuat tautan ke folder dataset dan folder model persisten. Jika path tujuan sudah berisi file/folder lain, sel akan berhenti dan tidak menghapus atau menimpanya.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

FMA_DRIVE_DIR = Path("/content/drive/MyDrive/fma_small")
DRIVE_MODELS_DIR = Path("/content/drive/MyDrive/appskripsi_models")

if not FMA_DRIVE_DIR.is_dir():
    raise FileNotFoundError(
        f"Folder dataset tidak ditemukan: {FMA_DRIVE_DIR}. Unggah FMA Small "
        "ke MyDrive/fma_small, termasuk folder audio dan tracks.csv atau "
        "fma_metadata.zip, lalu jalankan ulang sel ini."
    )

if next(FMA_DRIVE_DIR.glob("*/*.mp3"), None) is None:
    raise FileNotFoundError(
        f"Audio FMA Small (*.mp3) tidak ditemukan pada pola folder/berkas di {FMA_DRIVE_DIR}."
    )

if not (FMA_DRIVE_DIR / "tracks.csv").is_file() and not (FMA_DRIVE_DIR / "fma_metadata.zip").is_file():
    raise FileNotFoundError(
        f"Metadata FMA tidak ditemukan di {FMA_DRIVE_DIR}. Tambahkan tracks.csv "
        "atau fma_metadata.zip ke folder tersebut."
    )

DRIVE_MODELS_DIR.mkdir(parents=True, exist_ok=True)

def link_directory(link_path: Path, target_path: Path) -> None:
    if link_path.is_symlink():
        if link_path.resolve() == target_path.resolve():
            return
        raise FileExistsError(f"Tautan sudah ada dan mengarah ke lokasi lain: {link_path}")
    if link_path.exists():
        raise FileExistsError(f"Path sudah ada; tidak ditimpa: {link_path}")
    link_path.symlink_to(target_path, target_is_directory=True)

link_directory(REPO_DIR / "fma_small", FMA_DRIVE_DIR)
link_directory(REPO_DIR / "models", DRIVE_MODELS_DIR)
print("Dataset dan folder model Google Drive siap digunakan.")

## (Opsional) Latih model

Jalankan sel berikut hanya jika model belum tersedia di `MyDrive/appskripsi_models`. Ekstraksi fitur dan pelatihan dari seluruh FMA Small dapat berjalan lama. Hasil model akan tersimpan di Google Drive.

In [ ]:
RUN_MODEL_TRAINING = False  # Ubah ke True jika ingin mulai melatih model.

if RUN_MODEL_TRAINING:
    from ml_model import train_from_fma

    training_result = train_from_fma()
    print(training_result)
else:
    print("Pelatihan dilewati. Ubah RUN_MODEL_TRAINING ke True untuk melatih model.")

## Jalankan aplikasi Streamlit

Sel ini memulai aplikasi pada port 8501 dan membukanya di jendela baru. Biarkan runtime Colab aktif selama aplikasi digunakan.

In [ ]:
import time

log_path = Path("/content/streamlit.log")
log_file = log_path.open("w")
streamlit_process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "streamlit",
        "run",
        "app.py",
        "--server.port=8501",
        "--server.address=0.0.0.0",
        "--server.headless=true",
    ],
    cwd=REPO_DIR,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
time.sleep(5)

if streamlit_process.poll() is not None:
    log_file.close()
    raise RuntimeError(f"Streamlit gagal dimulai. Log:\n{log_path.read_text(errors='replace')}")

log_file.close()
from google.colab import output

output.serve_kernel_port_as_window(8501)